In [2]:
import pandas as pd
import numpy as np
import glob
import os
from PIL import Image
import pyvips as Vips

/home/mahirwar/miniconda3/envs/kfold_amy_plaque1/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/home/mahirwar/miniconda3/envs/kfold_amy_plaque1/lib/python3.9/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
dlb_wsi_dir = "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD_Dataset/LBD/DLB_cases"
dlb_wsi_dir1 = "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD_Dataset/LBD/DLB_cases/DLB_cases"
pdd_wsi_dir = "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD_Dataset/LBD/PDD_cases/PDD_cases"
pdd_wsi_dir1 = "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD_Dataset/LBD/PDD_cases/PDD_cases/PDD_CASES"

imagenames = glob.glob(os.path.join(dlb_wsi_dir, "*"))
print(len(imagenames))

234


In [3]:
imagenames = glob.glob(os.path.join(dlb_wsi_dir1, "*"))
print(len(imagenames))

220


In [4]:
234+220

454

In [5]:
234+220+290+32

776

In [6]:
imagenames = glob.glob(os.path.join(pdd_wsi_dir, "*"))
print(len(imagenames))

290


In [7]:
imagenames = glob.glob(os.path.join(pdd_wsi_dir1, "*"))
print(len(imagenames))

32


In [8]:
dlb_df = pd.DataFrame({"filename":imagenames})

In [9]:
len(imagenames)

32

In [12]:
file_corrupted = []
imagenames1 = glob.glob(os.path.join(dlb_wsi_dir, "*.svs"))
imagenames2 = glob.glob(os.path.join(dlb_wsi_dir1, "*.svs"))
imagenames3 = glob.glob(os.path.join(pdd_wsi_dir, "*.svs"))
imagenames4 = glob.glob(os.path.join(pdd_wsi_dir1, "*.svs"))
imagenames = sorted(imagenames1+imagenames2+imagenames3+imagenames4)

for img in imagenames:
    try:
        vips_img = Vips.Image.new_from_file(img, level=0)
    except:
        file_corrupted.append(img)

In [13]:
corrupted_df = pd.DataFrame({"corrupted":file_corrupted})

In [14]:
corrupted_df

,corrupted
0,/gladstone/finkbeiner/steve/work/data/npsad_da...
1,/gladstone/finkbeiner/steve/work/data/npsad_da...
2,/gladstone/finkbeiner/steve/work/data/npsad_da...
3,/gladstone/finkbeiner/steve/work/data/npsad_da...
4,/gladstone/finkbeiner/steve/work/data/npsad_da...
...,...
60,/gladstone/finkbeiner/steve/work/data/npsad_da...
61,/gladstone/finkbeiner/steve/work/data/npsad_da...
62,/gladstone/finkbeiner/steve/work/data/npsad_da...
63,/gladstone/finkbeiner/steve/work/data/npsad_da...


In [4]:
df = pd.DataFrame({"img_path":imagenames})

In [16]:
df = pd.merge(df,corrupted_df,how="left",left_on="img_path", right_on="corrupted" )

In [17]:
len(df)-df["corrupted"].isna().sum()

65

In [5]:
df["img_name"]=df["img_path"].apply(lambda l:l.split("/")[-1])

In [6]:
df

,img_path,img_name
0,/gladstone/finkbeiner/steve/work/data/npsad_da...,08_126_Syn1_CG_200x.svs
1,/gladstone/finkbeiner/steve/work/data/npsad_da...,04_103_Syn1_EntCx_200x.svs
2,/gladstone/finkbeiner/steve/work/data/npsad_da...,92_1434_Syn1_CG_200x.svs
3,/gladstone/finkbeiner/steve/work/data/npsad_da...,10_048_Syn1_EntCx_200x.svs
4,/gladstone/finkbeiner/steve/work/data/npsad_da...,99_1147_Syn1_FCx_200x.svs
...,...,...
215,/gladstone/finkbeiner/steve/work/data/npsad_da...,01_156_Syn1_CG_200x.svs
216,/gladstone/finkbeiner/steve/work/data/npsad_da...,97_1001_Syn1_EntCx_200x.svs
217,/gladstone/finkbeiner/steve/work/data/npsad_da...,02_186_Syn1_CG_200x.svs
218,/gladstone/finkbeiner/steve/work/data/npsad_da...,96_1014_Syn1_EntCx_200x.svs


In [19]:
count_img = df.groupby(["img_name"])["img_path"].count().reset_index()

In [20]:
count_img[count_img["img_path"]>1]

,img_name,img_path
231,16_009_CG_aSyn_x200.svs,2
232,16_009_EntCx_aSyn_x200.svs,2
233,16_009_FCx_aSyn_x200.svs,2
234,16_009_PCx_aSyn_x200.svs,2
235,16_009_TCx_aSyn_x200.svs,2
236,16_044_CG_aSyn_x200a.svs,2
325,18_031_FCx_aSyn_x200.svs,2
327,18_031_TCx_aSyn_x200.svs,2
328,18_056_EntCx_aSyn_x200.svs,2
329,18_056_FCx_aSyn_x200.svs,2


In [21]:
count_img.columns = ["img_name","count"]


In [22]:
df = pd.merge(df,count_img,on="img_name",how="left")

In [23]:
df[df["count"]>1].to_csv("/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/data_check/check_if_corrupted_files_reuploaded.csv")

In [24]:
df[(df["count"]==1) & (df["corrupted"].isna()==False)].to_csv("/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/data_check/corrupted_files.csv")

In [25]:
imagenames4 = glob.glob(os.path.join(pdd_wsi_dir1, "*.svs"))
#imagenames = sorted(imagenames1+imagenames2+imagenames3+imagenames4)
print(imagenames4)
corrupted2=[]
for img in imagenames4:
    try:
        vips_img = Vips.Image.new_from_file(img, level=0)
    except:
        corrupted2.append(img)

['/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD_Dataset/LBD/PDD_cases/PDD_cases/PDD_CASES/PD118_EntCx_Syn1.svs', '/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD_Dataset/LBD/PDD_cases/PDD_cases/PDD_CASES/PD118_CG_Syn1.svs', '/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD_Dataset/LBD/PDD_cases/PDD_cases/PDD_CASES/PD119_FCx_Syn1.svs', '/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD_Dataset/LBD/PDD_cases/PDD_cases/PDD_CASES/PD119_PCx_Syn1.svs', '/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD_Dataset/LBD/PDD_cases/PDD_cases/PDD_CASES/PD119_CG_Syn1.svs', '/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD_Dataset/LBD/PDD_cases/PDD_cases/PDD_CASES/PD199_Syn1_CG.svs', '/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD_Dataset/LBD/PDD_cases/PDD_cases/PDD_CASES/PD118_PCx_Syn1.svs', '/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD_Dataset/LBD/PDD_cases/PDD_cases/PDD_CASES/PD180_EntCx_Syn1.svs', '/glad

In [26]:
df[df["img_path"].isin(imagenames4)]

,img_path,corrupted,img_name,count
741,/gladstone/finkbeiner/steve/work/data/npsad_da...,NaN,PD092_SYN1_CG.svs,1
742,/gladstone/finkbeiner/steve/work/data/npsad_da...,NaN,PD092_SYN1_ENTCX.svs,1
743,/gladstone/finkbeiner/steve/work/data/npsad_da...,NaN,PD092_SYN1_FCX.svs,1
744,/gladstone/finkbeiner/steve/work/data/npsad_da...,NaN,PD092_SYN1_PCX.svs,1
745,/gladstone/finkbeiner/steve/work/data/npsad_da...,NaN,PD092_TCx_Syn1.svs,1
746,/gladstone/finkbeiner/steve/work/data/npsad_da...,NaN,PD102_Syn1_CG.svs,1
747,/gladstone/finkbeiner/steve/work/data/npsad_da...,NaN,PD113_TCx_Syn1.svs,1
748,/gladstone/finkbeiner/steve/work/data/npsad_da...,NaN,PD118_CG_Syn1.svs,1
749,/gladstone/finkbeiner/steve/work/data/npsad_da...,NaN,PD118_EntCx_Syn1.svs,1
750,/gladstone/finkbeiner/steve/work/data/npsad_da...,NaN,PD118_FCx_Syn1.svs,1


In [7]:
df["Autopsy no"]=df["img_name"].apply(lambda l:l.split("_")[0] if l.startswith("PD") else l.split("_")[0]+"/"+l.split("_")[1])

In [8]:
df

,img_path,img_name,Autopsy no
0,/gladstone/finkbeiner/steve/work/data/npsad_da...,08_126_Syn1_CG_200x.svs,08/126
1,/gladstone/finkbeiner/steve/work/data/npsad_da...,04_103_Syn1_EntCx_200x.svs,04/103
2,/gladstone/finkbeiner/steve/work/data/npsad_da...,92_1434_Syn1_CG_200x.svs,92/1434
3,/gladstone/finkbeiner/steve/work/data/npsad_da...,10_048_Syn1_EntCx_200x.svs,10/048
4,/gladstone/finkbeiner/steve/work/data/npsad_da...,99_1147_Syn1_FCx_200x.svs,99/1147
...,...,...,...
215,/gladstone/finkbeiner/steve/work/data/npsad_da...,01_156_Syn1_CG_200x.svs,01/156
216,/gladstone/finkbeiner/steve/work/data/npsad_da...,97_1001_Syn1_EntCx_200x.svs,97/1001
217,/gladstone/finkbeiner/steve/work/data/npsad_da...,02_186_Syn1_CG_200x.svs,02/186
218,/gladstone/finkbeiner/steve/work/data/npsad_da...,96_1014_Syn1_EntCx_200x.svs,96/1014


In [9]:
df["syn1"] = df["img_name"].apply(lambda l: l.lower().find("syn1")!=-1, 1, 0)

In [12]:
df[["Autopsy no",	"syn1"]].drop_duplicates().to_csv("/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/data_check/autopsy_img_name_mapping.csv")

In [29]:
df["CG"] = df["img_name"].apply(lambda l: 1 if (l.find("CG")!=-1) else 0)

In [30]:
df["EntCx"] = df["img_name"].apply(lambda l: 1 if (l.lower().find("entcx")!=-1) else 0)

In [31]:
df["FCx"] = df["img_name"].apply(lambda l: 1 if (l.lower().find("fcx")!=-1) else 0)

In [32]:
df["TCx"] = df["img_name"].apply(lambda l: 1 if ((l.lower().find("entcx")==-1) & (l.lower().find("tcx")!=-1)) else 0)

In [33]:
df["PCx"] = df["img_name"].apply(lambda l: 1 if (l.lower().find("pcx")!=-1) else 0)

In [34]:
df

,img_path,corrupted,img_name,count,Autopsy no,CG,EntCx,FCx,TCx,PCx
0,/gladstone/finkbeiner/steve/work/data/npsad_da...,NaN,11_063_CG_aSyn_x200.svs,1,11/063,1,0,0,0,0
1,/gladstone/finkbeiner/steve/work/data/npsad_da...,NaN,11_063_EntCx_aSyn_x200.svs,1,11/063,0,1,0,0,0
2,/gladstone/finkbeiner/steve/work/data/npsad_da...,NaN,11_063_FCx_aSyn_x200.svs,1,11/063,0,0,1,0,0
3,/gladstone/finkbeiner/steve/work/data/npsad_da...,NaN,11_063_PCx_aSyn_x200.svs,1,11/063,0,0,0,0,1
4,/gladstone/finkbeiner/steve/work/data/npsad_da...,NaN,11_063_TCx_aSyn_x200.svs,1,11/063,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...
767,/gladstone/finkbeiner/steve/work/data/npsad_da...,NaN,PD250_PCx_Syn1.svs,1,PD250,0,0,0,0,1
768,/gladstone/finkbeiner/steve/work/data/npsad_da...,NaN,PD501_EntCx_Syn1.svs,1,PD501,0,1,0,0,0
769,/gladstone/finkbeiner/steve/work/data/npsad_da...,NaN,PD531_CG_Syn.svs,1,PD531,1,0,0,0,0
770,/gladstone/finkbeiner/steve/work/data/npsad_da...,NaN,PD531_FCx_Syn.svs,1,PD531,0,0,1,0,0


In [35]:
df_new = df[["Autopsy no","CG","EntCx","FCx","TCx","PCx"]]

In [36]:
df["img_name"].nunique()

761

In [37]:
agg_index = {"CG":sum,"EntCx":sum,"FCx":sum,"TCx":sum,"PCx":sum }
df_new1= df_new.groupby(["Autopsy no"]).agg(agg_index).reset_index()

In [51]:
df_new1[df_new1["CG"]>1]

,Autopsy no,CG,EntCx,FCx,TCx,PCx
74,16/009,2,2,2,2,2
75,16/044,3,1,1,1,1
76,16/050,2,1,1,1,1
77,16/079,2,1,1,1,1
83,17/010,2,2,2,2,2
149,PD092,2,2,2,2,2
153,PD118,2,2,2,2,2
154,PD119,2,2,2,2,2
171,PD182,2,2,1,1,1
202,PD531,2,1,2,0,2


In [52]:
df_new1[df_new1["EntCx"]>1]

,Autopsy no,CG,EntCx,FCx,TCx,PCx
54,12/097,1,2,1,1,1
74,16/009,2,2,2,2,2
83,17/010,2,2,2,2,2
93,18/056,0,2,2,1,2
149,PD092,2,2,2,2,2
153,PD118,2,2,2,2,2
154,PD119,2,2,2,2,2
169,PD179,1,2,1,2,1
170,PD180,1,2,2,0,1
171,PD182,2,2,1,1,1


In [53]:
df_new1[df_new1["FCx"]>1]

,Autopsy no,CG,EntCx,FCx,TCx,PCx
69,15/072,1,1,2,1,1
74,16/009,2,2,2,2,2
83,17/010,2,2,2,2,2
92,18/031,1,1,2,2,1
93,18/056,0,2,2,1,2
149,PD092,2,2,2,2,2
153,PD118,2,2,2,2,2
154,PD119,2,2,2,2,2
170,PD180,1,2,2,0,1
202,PD531,2,1,2,0,2


In [54]:
df_new1[df_new1["PCx"]>1]

,Autopsy no,CG,EntCx,FCx,TCx,PCx
74,16/009,2,2,2,2,2
83,17/010,2,2,2,2,2
93,18/056,0,2,2,1,2
149,PD092,2,2,2,2,2
153,PD118,2,2,2,2,2
154,PD119,2,2,2,2,2
182,PD250,1,1,1,1,2
202,PD531,2,1,2,0,2


In [55]:
df_new1[df_new1["TCx"]>1]

,Autopsy no,CG,EntCx,FCx,TCx,PCx
74,16/009,2,2,2,2,2
83,17/010,2,2,2,2,2
84,17/021,1,1,1,2,1
92,18/031,1,1,2,2,1
149,PD092,2,2,2,2,2
152,PD113,1,0,1,2,1
153,PD118,2,2,2,2,2
154,PD119,2,2,2,2,2
169,PD179,1,2,1,2,1


In [38]:
orig_list = "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Jiayi_files/PDD_DLB_1989-2019_updated.csv"
orig_list = pd.read_csv(orig_list)

In [39]:
orig_list

,Autopsy no,Gender,Age,Path_Diagnosis,CG,EntCx,FCx,TCx,PCx
0,00/1027,NaN,NaN,DLB+AD,1.0,1.0,1.0,1.0,1.0
1,00/1108,NaN,NaN,DLB+AD,1.0,1.0,1.0,1.0,1.0
2,00/1140,NaN,NaN,pure DLB,1.0,1.0,1.0,1.0,1.0
3,01/104,NaN,NaN,DLB+AD,1.0,1.0,1.0,1.0,1.0
4,01/156,NaN,NaN,DLB+AD,1.0,1.0,1.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...
202,PD537,M,84,PDD,1.0,1.0,1.0,1.0,1.0
203,PD538,M,73,PDD,1.0,1.0,0.0,1.0,1.0
204,PD544,M,81,PDD,1.0,1.0,1.0,1.0,1.0
205,PD553,M,77,PDD+AD,1.0,1.0,1.0,1.0,1.0


In [40]:
orig_list1 = pd.merge(orig_list,df_new1, on="Autopsy no",how="outer" )

In [41]:
orig_list1[:-50]

,Autopsy no,Gender,Age,Path_Diagnosis,CG_x,EntCx_x,FCx,TCx,PCx,CG_y,EntCx_y,FCx,TCx,PCx
0,00/1027,NaN,NaN,DLB+AD,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
1,00/1108,NaN,NaN,DLB+AD,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0
2,00/1140,NaN,NaN,pure DLB,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0
3,01/104,NaN,NaN,DLB+AD,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
4,01/156,NaN,NaN,DLB+AD,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
177,PD246,M,81,PDD,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
178,PD250,M,81,PDD,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,2.0
179,PD258,M,69,PDD,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0
180,PD264,F,83,PDD+AD,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0


In [42]:
orig_list1.to_csv("/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/data_check/original_data_received_files_flagged2.csv")

In [43]:
orig_list1.columns

Index(['Autopsy no', 'Gender', 'Age', 'Path_Diagnosis', 'CG_x', 'EntCx_x',
       'FCx ', 'TCx ', 'PCx ', 'CG_y', 'EntCx_y', 'FCx', 'TCx', 'PCx'],
      dtype='object')

## Find duplicates

In [49]:
cg_count = df[df["CG"]==1].groupby(["Autopsy no","CG"])["img_name"].count()
cg_count = df[df["CG"]==1].groupby(["Autopsy no","CG"])["img_name"].count()
cg_count = df[df["CG"]==1].groupby(["Autopsy no","CG"])["img_name"].count()
cg_count = df[df["CG"]==1].groupby(["Autopsy no","CG"])["img_name"].count()
cg_count = df[df["CG"]==1].groupby(["Autopsy no","CG"])["img_name"].count()

Autopsy no  CG
00/1027     1     1
00/1108     1     1
01/104      1     1
01/156      1     1
02/019      1     1
                 ..
PD341       1     1
PD430       1     1
PD493       1     1
PD501       1     1
PD531       1     2
Name: img_name, Length: 176, dtype: int64

## All Syn1 cases

In [54]:
path  = "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/Antibodies_data/Syn1"

In [55]:

imagenames = glob.glob(os.path.join(path, "*.svs"))

In [56]:
imagenames

['/gladstone/finkbeiner/steve/work/data/npsad_data/monika/Antibodies_data/Syn1/PDD_PD471_PCx_Syn1.svs',
 '/gladstone/finkbeiner/steve/work/data/npsad_data/monika/Antibodies_data/Syn1/PDD_PD550_CGx_Syn1.svs',
 '/gladstone/finkbeiner/steve/work/data/npsad_data/monika/Antibodies_data/Syn1/PDD_PD255_EntCx_Syn1.svs',
 '/gladstone/finkbeiner/steve/work/data/npsad_data/monika/Antibodies_data/Syn1/PDD_PD356_EntCx_Syn1.svs',
 '/gladstone/finkbeiner/steve/work/data/npsad_data/monika/Antibodies_data/Syn1/PDD_PD294_CGx_Syn1.svs',
 '/gladstone/finkbeiner/steve/work/data/npsad_data/monika/Antibodies_data/Syn1/PDD_PD354_CGx_Syn1.svs',
 '/gladstone/finkbeiner/steve/work/data/npsad_data/monika/Antibodies_data/Syn1/PDD_PD356_CGx_Syn1.svs',
 '/gladstone/finkbeiner/steve/work/data/npsad_data/monika/Antibodies_data/Syn1/PDD_PD330_EntCx_Syn1.svs',
 '/gladstone/finkbeiner/steve/work/data/npsad_data/monika/Antibodies_data/Syn1/PDD_PD498_FCx_Syn1.svs',
 '/gladstone/finkbeiner/steve/work/data/npsad_data/monika/

In [57]:
path2 = "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD_Dataset/LBD/DLB_cases/"

In [58]:
imagenames1 = glob.glob(os.path.join(path2, "*.svs"))

In [59]:
path3 = "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD_Dataset/LBD/PDD_cases/PDD_cases/"

In [60]:
imagenames2 = glob.glob(os.path.join(path3, "*.svs"))

In [61]:
len(imagenames2)

288

In [62]:
path4 = "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD_Dataset/LBD/PDD_cases/PDD_cases/PDD_CASES"

In [63]:
imagenames3 = glob.glob(os.path.join(path4, "*.svs"))

In [64]:
len(imagenames3)

31

In [65]:
path5 = "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD_Dataset/David-DLB_cases/"

In [66]:
imagenames4 = glob.glob(os.path.join(path5, "*"))

In [67]:
imagenames4

['/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD_Dataset/David-DLB_cases/PD221-1_Syn1_FCx David Menassa.svs',
 '/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD_Dataset/David-DLB_cases/93_1075_Syn1_CG David Menassa.svs',
 '/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD_Dataset/David-DLB_cases/93_1135_Syn1_EntCx David Menassa.svs',
 '/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD_Dataset/David-DLB_cases/92_1434_Syn1_TCx David Menassa.svs',
 '/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD_Dataset/David-DLB_cases/95_1050_Syn1_PCx David Menassa.svs',
 '/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD_Dataset/David-DLB_cases/92_1434_Syn1_FCx David Menassa.svs',
 '/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD_Dataset/David-DLB_cases/PD246_Syn1_PCx David Menassa.svs',
 '/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD_Dataset/David-DLB_cases/92_1357_Syn1_TCx David Menassa.svs',
 '/gladstone/fink

In [69]:
all_list  = imagenames+imagenames1+imagenames2+imagenames3

In [71]:
import pandas as pd
df = pd.DataFrame({"filename":all_list})

In [73]:
df["pat_id"] = df["filename"].apply(lambda l: l.split("/")[-1])

In [76]:
df.to_csv("/gladstone/finkbeiner/steve/work/data/npsad_data/monika/syn1_files_received.csv")